# 0. Upstage API KEY 발급

## 0-1. 쿠폰 발급 방법 및 기간

### 0-1-1. 쿠폰 정보

1. 사용자 쿠폰 코드: `SSAFY_AI_2_2025_10`
2. 발급 금액: $50 상당의 Credit
3. 등록 가능 기간: **~ 2025년 10월 30일(수)**

### 0-1-2. 쿠폰 등록 방법

1. 링크 접속 및 회원 가입
    - https://console.upstage.ai/docs/getting-started
    - `구글로 로그인`
2. 상단 메뉴에서 Dashboard → Billing → Credit → Redeem code 선택
3. 위 사용자 쿠폰 코드를 입력 후 등록 완료

### 0-1-3. KEY 확인

-  `Dashboard`에서 발급된 키 확인
    - 쿠폰을 포함한 KEY 값 모두 `절대 외부 유출 금지`

# 1. 환경설정 및 핵심 라이브러리

- LLM API를 활용한 데이터 생성 및 증강 작업을 진행할 예정
- API 키를 안전하게 관리하고, 효율적인 통신을 위한 기본 설정이 필요

## 1-1. 환경 변수

### 1-1-1.환경 변수란?

- 운영체제가 프로그램 실행 환경에 제공하는 전역 설정값
- 프로그램은 이 변소를 통해 API 키, DB 비밀번호 등 민감한 정보를 코드 외부에서 읽어 올 수 있음.

### 1-1-2. 환경 변수 설정 이유와 방법

1. **보안**
    - API KEY와 같은 민감 정보를 코드에 작성하고, github 등에 공유 시 유출의 위험이 있음.
    - 환경 변수를 통해 민감 정보를 코드와 분리하여 안전하게 관리. 단, 이 방법이 완벽한 보안을 보장하는 것은 아니므로 주의가 필요함
2. **유연성**
    - 개발, 테스트, 배포 등 각 환경마다 다른 API KEY를 사용해야 할 경우, 코드를 직접 수정하지 않고 환경 변수 값만 변경하여 유연하게 관리할 수 있음
3. **코드 작성**
    - `dotenv` 라이브러리: `.env` 파일에 저장된 키-값 쌍을 운영체제의 환경 변수로 불러오는 역할을 함
    - `os.getenv`: 로드된 환경 변수 중에서 "지정한 키"에 해당하는 값을 파이썬 코드 내로 가져옴
    

In [ ]:
# 구글 드라이브를 코랩 환경에 마운트.
# 이를 통해 드라이브에 저장된 파일(.env 등)에 접근 가능.
from google.colab import drive
drive.mount('/content/drive')

# API 키 파일이 저장된 기본 경로를 설정.
base_path = '/content/drive/MyDrive/Colab Notebooks/AI/08_Data_Synthesis/'

In [ ]:
# Colab 환경에서 .env 파일을 생성하고 API 키를 저장하는 명령어.
# 실제 키를 {your_api_key} 부분에 입력
!echo "UPSTAGE_API_KEY={your_api_key}" > "/content/drive/MyDrive/Colab Notebooks/AI/08_Data_Synthesis/.env"

In [ ]:
# .env 파일에서 환경 변수를 로드하기 위한 라이브러리.
from dotenv import load_dotenv
# 운영체제의 환경 변수를 가져오기 위한 함수.
from os import getenv

# .env 파일을 로드하여 환경 변수를 설정.
load_dotenv(base_path + ".env")

# getenv 함수를 사용해 "UPSTAGE_API_KEY"라는 이름의 환경 변수 값을 가져옴.
UPSTAGE_API_KEY = getenv("UPSTAGE_API_KEY")

# API 키가 성공적으로 로드되었는지 확인하고 메시지를 출력.
if UPSTAGE_API_KEY:
    print("Success API Key Setting!")
else:
    print(f"ERROR: Failed to load UPSTAGE_API_KEY from {base_path}")
    

## 1-2. Upstage API 기본 사용 방법

### 1-2-1. openai 라이브러리 사용

- Upstage API는 OpenAI API와 요청 및 응답 구조가 호환됨.
- 따라서, `openai` 라이브러리를 그대로 활용하되, 요청을 보내는 서버의 주소(`base_url`)만 Upstage API 엔드포인트로 변경하여 사용.

In [ ]:
!pip install openai

In [ ]:
# openai 라이브러리에서 OpenAI 클라이언트 클래스를 임포트.
from openai import OpenAI

# Upstage API와 통신하기 위한 클라이언트를 생성.
client = OpenAI(
    # 환경 변수에서 읽어온 API 키를 사용.
    api_key=UPSTAGE_API_KEY,
    # API 요청을 보낼 기본 URL을 Upstage 서버 주소로 지정.
    base_url="https://api.upstage.ai/v1",
)

# client.chat.completions.create: 채팅 기반의 LLM 응답을 생성하는 메서드.
stream = client.chat.completions.create(
    # model: 사용할 LLM 모델을 지정.
    model="solar-pro2",
    # messages: 모델에게 전달할 대화 내용. 역할(role)과 내용(content)으로 구성.
    messages=[
        {
            "role": "user",
            "content": "안녕? 넌 이름이 뭐니?",
        }
    ],
    # stream=True: 응답을 한 번에 받지 않고, 생성되는 대로 조각(chunk) 단위로 실시간 수신.
    stream=True,
)

# 스트리밍된 응답을 실시간으로 처리.
for chunk in stream:
    # 각 조각(chunk)에 새로운 텍스트 내용이 있는지 확인.
    if chunk.choices[0].delta.content is not None:
        # end=""는 줄바꿈 없이 텍스트를 이어서 출력하게 함.
        print(chunk.choices[0].delta.content, end="")

# stream=False로 설정했을 경우, 아래와 같이 전체 응답을 한 번에 확인 가능.
# print(stream.choices[0].message.content)

## 1-3. HTTPX (비동기 통신)

### 1-3-1. HTTPX란?

- Python용 HTTP 클라이언트 라이브러리
- 기존 `requests` 라이브러리와 유사한 사용법을 제공하면서 **비동기(Asynchronous)** 통신을 지원하는 것이 가장 큰 특징.
- **비동기 통신:** 여러 개의 API 요청을 보낼 때, 하나의 요청이 끝날 때까지 기다리지 않고 여러 요청을 **동시에 병렬적으로 처리**하는 방식. 이를 통해 전체 작업 시간을 크게 단축시킬 수 있음.


### 1-3-2. 코드 작성

- `async def`: 함수가 **비동기 함수**임을 선언. 이 함수 내에서는 `await` 키워드를 사용할 수 있음.
- `await`: 비동기 작업(예: API 요청)이 완료될 때까지 기다리면서, 프로그램의 다른 비동기 작업은 계속 실행되도록 함. 즉, **"기다리는 동안 다른 일 먼저 하고 있어!"** 라는 의미.

In [ ]:
import httpx
import asyncio

# 비동기적으로 Upstage API를 호출하는 함수를 정의.
async def call_chat_completion(url, headers, payload):
    print('    call_chat_completion 시작')
    # httpx.AsyncClient를 사용하여 비동기 HTTP 요청 클라이언트를 생성.
    # timeout=30.0: 30초 동안 응답이 없으면 타임아웃 오류를 발생시킴.
    async with httpx.AsyncClient(timeout=30.0) as client:
        # await client.post: API에 POST 요청을 보내고 응답이 올 때까지 비동기적으로 대기.
        response = await client.post(url, headers=headers, json=payload)
        # HTTP 오류(4xx, 5xx)가 발생하면 예외를 발생시킴.
        response.raise_for_status()
        data = response.json()

        # 응답 데이터에서 실제 생성된 텍스트 메시지를 반환.
        return data["choices"][0]["message"]["content"]

In [ ]:
# 여러 비동기 작업을 실행하고 결과를 처리하는 메인 함수.
async def request(tasks):
    print('이제 각 요청 실행 시작')
    # asyncio.gather(*tasks): 리스트에 담긴 모든 비동기 작업을 동시에 실행하고,
    # 모든 작업이 완료될 때까지 기다린 후 결과를 리스트로 모아서 반환.
    results = await asyncio.gather(*tasks)
    print('모든 요청 완료')
    print()

    # 완료된 결과를 하나씩 출력.
    for i, res in enumerate(results, 1):
        print(f'Response {i}')
        print(res)
        print('=' * 20)


# API 엔드포인트 및 인증 헤더 설정.
url = "https://api.upstage.ai/v1/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {UPSTAGE_API_KEY}",
}

# 동시에 보낼 여러 개의 프롬프트를 리스트로 준비.
prompts = [
    "농담 하나만 해 줘",
    "프랑스의 수도는 어디야?",
]

# 실행할 비동기 작업들을 담을 리스트.
tasks = []
# 각 프롬프트에 대해 API 요청 페이로드를 생성.
for prompt in prompts:
    payload = {
        "model": "solar-pro2",
        "messages": [
            {
                "role": "user",
                "content": prompt,
            }
        ],
        # stream=False: 전체 응답을 한 번에 받도록 설정.
        "stream": False,
    }
    # call_chat_completion 함수를 호출하여 비동기 작업(Task) 객체를 생성하고 리스트에 추가.
    # 'await'가 없으므로 함수가 바로 실행되지 않고, 실행 대기 상태의 객체만 만들어짐.
    print('태스크 생성')
    tasks.append(call_chat_completion(url, headers, payload))
    print('아직 upstage API 호출 전')

# 준비된 모든 태스크를 동시에 실행.
await request(tasks)

### 1-4. JSON (JavaScript Object Notation)

- 데이터를 저장하거나 주고받을 때 사용하는 가볍고 사람이 읽기 쉬운 데이터 형식.

- 파이썬의 딕셔너리와 유사한 `"키": 값` 형태의 쌍으로 구성됨.

- 대부분의 프로그래밍 언어와 API 통신에서 표준처럼 사용됨.

- LLM에게 답변을 정해진 형식으로 받기 위해 `response_format` 옵션을 사용하여 출력을 JSON 구조로 강제할 수 있음.

In [ ]:
import json

# LLM의 응답을 구조화된 JSON 형식으로 강제하기 위한 설정.
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "수도 정보",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "capital": {"type": "string"},
                "translation": {"type": "string", "description": "수도의 영어 번역"},
            },
            "required": ["capital", "translation"],
        },
    },
}

# client.chat.completions.create 메서드를 호출하여 LLM에 요청.
response = client.chat.completions.create(
    model="solar-pro2",
    messages=[
        {
            "role": "user",
            "content": "한국의 수도는 어디야?",
        }
    ],
    # 위에서 정의한 JSON 스키마를 적용하여 응답 형식을 강제.
    response_format=response_format,
)

# LLM의 응답은 JSON 형식의 '문자열'이므로,
# json.loads를 사용하여 파이썬 딕셔너리 객체로 변환.
structured_dictionary = json.loads(response.choices[0].message.content)

# 딕셔너리로 변환된 구조화된 응답을 출력.
print("Structured Response:")
for key, value in structured_dictionary.items():
    print(f"{key}: {value}")

- 더 구체적인 response_format 사용 방법은 아래 링크 참고
- [openai Structured model outputs](https://platform.openai.com/docs/guides/structured-outputs?lang=python)